In [ ]:
"""
Toy CMB TT angular power spectrum following the structure in Cosm9.pdf:

- δT/T has 3 contributions:
    (1) Effective temperature:  (Θ0 + Φ)  ~ 1/4 δγ + Φ
    (2) Doppler:               -v · n̂     (with Θ1 ~ v/3)
    (3) ISW:                    2∫ Φ̇ dt
  plus cross-terms in C_l (quadratic quantity).

- Projection: k -> l via spherical Bessel functions j_l(k d_A),

approximation.
"""

import numpy as np
import matplotlib.pyplot as plt
import camb
from camb import initialpower

pars = camb.CAMBparams()
pars.set_cosmology(H0=67.5, ombh2=0.022, omch2=0.122, mnu=0.06, omk=0.0, tau=0.06)
pars.InitPower.set_params(ns=0.965, As=2e-9)
pars.set_for_lmax(2000, lens_potential_accuracy=1)

results = camb.get_results(pars)

#CMB spectra 
powers = results.get_cmb_power_spectra(pars, CMB_unit='muK')

# Lensed and unlensed spectra
lensed   = powers['total']           
unlensed = powers['unlensed_scalar']

ells = np.arange(lensed.shape[0])






import numpy as np
import matplotlib.pyplot as plt
from scipy.special import spherical_jn


# Multipole range
lmin = 2
lmax = 2000

# Primordial curvature spectrum P_R(k) = A_s (k/kp)^(n_s-1), where As is taken from Planck
# dimensionless variable x = k d_A, so PR(x) uses a pivot xp = kp d_A
As = 2*1e-9
ns = 0.96

# Acoustic scale in multipole space: l_A = π d_A / r_s  
# Typical observed is ~300; treating this as a tuning knob.
lA = 300.0
theta_s = np.pi / lA  # = r_s / d_A  

# Diffusion damping scale in l-space: exp[-l^2/l_D^2] 
lD = 300.0

# A rough "radiation driving / early ISW" boost near the first peak
drive_amp = 0.1
drive_scale = 100.0

# ISW source: mostly low-l / around first peak in this toy model (early ISW only)
isw_amp = 0.05
isw_scale = 900.0

# Doppler overall relative strength (the PDF notes Doppler is subdominant and fills troughs)
doppler_amp = 0.55

# Cross-term scaling (keep =1 unless you want to exaggerate/suppress)
cross_scale = 1.0

# Numerical integration grid in x = k d_A (dimensionless)
# j_l(x) peaks around x ~ l, so we integrate out past lmax by a margin.
x_max = lmax + 3000
dx = 1.0
x = np.arange(1.0, x_max + dx, dx)  # start at 1 to avoid x=0 singularities in dk/k weighting

# Pivot in x-space (only matters if ns != 1)
xp = 0.05 * 14000.0  # kp*d_A; choose d_A~14000 Mpc just to set a pivot scale in x

#Primordial spectrum in x-variable
# With x = k d_A: dk/k = dx/x and PR(k) -> PR(x) = As^2 (x/xp)^(ns-1)
PR = (As) * (x / xp) ** (ns - 1.0)

#Source / transfer functions 
# Damping factor ~ exp[-(k/kD)^2] becomes exp[-(x/xD)^2], and since x~l we can use lD as xD.
damp = np.exp(-(x / lD) ** 2)

# "Driving" enhancement (bumps first peak region)
drive = 1.0 + drive_amp * np.exp(-(x / drive_scale) ** 2)

# Effective temperature source S_T(x) ~ (Θ0+Φ):
# acoustic cosine phase 
# Add an explicit ordinary Sachs–Wolfe plateau piece 
# From the notes: (δT/T)_SW = -(1/5) R  (superhorizon)


# SW should only contribute on superhorizon/large scales.
# Choose xSW ~ O(50–100) so it dies away before the acoustic region dominates.
#xSW = 100.0
#Wsw = np.exp(-(x/xSW)**2)   # 1 at low x, ->0 at high x
#S_SW = -0.2 * Wsw

S_SW = -0.2 * np.ones_like(x) * damp  # -1/5 = -0.2 (sets the plateau normalization)


# Turn on acoustic oscillations only after horizon entry (toy window) 
# Choose a transition scale in x ~ l_H 
xH = 200.0
W = 1.0 - np.exp(-(x / xH)**2)  # W→0 for x<<xH, W→1 for x>>xH


# Acoustic effective temperature part (subhorizon)
# Total effective-temperature source: SW plateau + acoustic part
S_ac = drive * np.cos(x * theta_s) * damp
ST  = S_SW + W * S_ac

# Doppler source S_V(x) ~ v (out of phase: sine)
SV = doppler_amp * drive * np.sin(x * theta_s) * damp

# ISW source S_I(x): low-x dominated 
SI = isw_amp * np.exp(-(x / isw_scale) ** 2) 


#Compute C_l contributions via line-of-sight projection
# For SW-like terms: integrand ~ PR(x) * S(x)^2 * j_l(x)^2 * dx/x
# For Doppler: it projects roughly with derivative of j_l 
# j_l'(x) using spherical_jn(l, x, derivative=True).

ls = np.arange(lmin, lmax + 1)

Cl_T  = np.zeros_like(ls, dtype=float)  # (Θ0+Φ)^2
Cl_V  = np.zeros_like(ls, dtype=float)  # Doppler^2
Cl_I  = np.zeros_like(ls, dtype=float)  # ISW^2

Cl_TV = np.zeros_like(ls, dtype=float)  # (Θ0+Φ)×Doppler cross
Cl_IT = np.zeros_like(ls, dtype=float)  # ISW×(Θ0+Φ) cross
Cl_IV = np.zeros_like(ls, dtype=float)  # ISW×Doppler cross

# Common weight for dk/k -> dx/x
w = (dx / x) * PR


for i, ell in enumerate(ls):
    jl  = spherical_jn(ell, x)
    jlp = spherical_jn(ell, x, derivative=True)

    # Auto terms
    Cl_T[i] = 4.0 * np.pi * np.sum(w * (ST**2) * (jl**2))
    Cl_V[i] = 4.0 * np.pi * np.sum(w * (SV**2) * (jlp**2))
    Cl_I[i] = 4.0 * np.pi * np.sum(w * (SI**2) * (jl**2))

    # Cross terms: 2 * ∫ PR * S_a * S_b * projection_a * projection_b
    # ((Θ0+Φ)×Doppler is small because nearly out of phase.)
    Cl_TV[i] =  cross_scale * 2.0 * 4.0 * np.pi * np.sum(w * (ST * SV) * (jl * jlp)) #Effective cross doppler
    Cl_IT[i] = cross_scale * 2.0 * 4.0 * np.pi * np.sum(w * (SI * ST) * (jl * jl)) #ISW Cross term
    Cl_IV[i] =  cross_scale * 2.0 * 4.0 * np.pi * np.sum(w * (SI * SV) * (jl * jlp))

# Full spectrum
Cl_full = Cl_T + Cl_V + Cl_I + Cl_TV + Cl_IT + Cl_IV

# D_l{TT} = T^2 l(l+1) C_l / (2π)
T0_uK = 2.725e6   # μK
TT = T0_uK**2  #(μK)^2
pref = (ls * (ls + 1)) /(2* np.pi) * TT
Dl_full = pref * Cl_full
Dl_T    = pref * Cl_T
Dl_V    = pref * Cl_V
Dl_I    = pref * Cl_I
Dl_ISW_cross = pref * (Cl_IT + Cl_IV)
Dl_TV_cross  = pref * Cl_TV


plt.figure(figsize=(10, 6))
plt.plot(ls, Dl_full, label="Full spectrum")
plt.plot(ls, Dl_T,    label=r"$\Theta_0+\Phi$ (effective temperature)")
plt.plot(ls, Dl_V,    label=r"Doppler ($\Theta_1$)")
plt.plot(ls, Dl_I,    label="ISW")
plt.plot(ls, Dl_ISW_cross, label="ISW cross terms")
plt.plot(ls, Dl_TV_cross,  label=r"$(\Theta_0+\Phi)\times$ Doppler")
plt.plot(ells, lensed[:, 0],  label="TT lensed", color="black")

plt.xlim(lmin, lmax)
plt.ylim(bottom=-500)
#plt.xscale("log")
plt.xlabel(r"Multipole $\ell$")
plt.ylabel(r"$\ell(\ell+1)C_\ell^{TT}/(2\pi)(μK^2)$")
plt.title("Toy CMB TT power spectrum")
plt.legend()
plt.tight_layout()
plt.show()


# - lA controls peak spacing (acoustic scale).
# - lD controls high-l damping tail.
# - drive_amp/drive_scale boosts the first peak region (early ISW / radiation driving).
# - doppler_amp sets how much trough-filling you get.


In [ ]:
from scipy.signal import find_peaks

# --- Find the first three acoustic peaks in D_l ---
# (Ignore the very-low-l SW plateau by starting the search at e.g. l >= 30–50)
l_start = 30
mask = ls >= l_start

# Find local maxima; tune these if you get spurious peaks
peaks, props = find_peaks(
    Dl_full[mask],
    prominence=np.max(Dl_full[mask]) * 0.01,  # 1% prominence
    distance=80                                # enforce separation in l
)

# Convert peak indices back to full-l indices
peak_ls = ls[mask][peaks]
peak_vals = Dl_full[mask][peaks]

# Sort by l and take first three
order = np.argsort(peak_ls)
peak_ls = peak_ls[order][:3]
peak_vals = peak_vals[order][:3]

print("First three peak locations (l):")
for i, (ell, val) in enumerate(zip(peak_ls, peak_vals), start=1):
    print(f"Peak {i}: l = {ell},  D_l = {val:.4g}")

In [ ]:
print("damp(high x) ~", damp[-1])
print("W(high x) ~", W[-1])
print("ST(high x) ~", ST[-1])